# 🏢 Ghana Building Footprint Extraction — Stage 2 (SegFormer-B3)

**Notebook version: v3 (2026-08-24)** — the run-3 notebook. Changelog:
- **v1** (run-2, Kaggle T4): v2 recipe on the v1 dataset (val = AHAFO only).
  Result: val IoU 0.4613 @ epoch 1, then collapse — same trajectory as U-Net
  run-1. Diagnosed as val-split distribution shift, NOT a recipe problem.
- **v3 (this one)**: weight decay 1e-4 → **1e-2**; dataset v1 → **v2**
  (`ghana-buildings-patches-v2`: AHAFO in train, multi-region spatial-block
  val, matched densities train 0.200 / val 0.176). Everything else unchanged.

**HOT fAIr GeoAI Open Call submission — main model.** Uses the fixed v2 recipe proven-needed by the U-Net run-1 diagnosis (val IoU peaked at epoch 1 = 0.4495 with old recipe).

| Item | Value |
|---|---|
| Task | Binary building footprint segmentation |
| Data | 15 Ghana regions, 0.30 m/px VHR RGB, Overture labels |
| Splits (v2) | train: 12 regions incl. AHAFO (3,272 chips) · val: multi-region spatial blocks, 10 regions (424) · test: NORTHERN/VOLTA/WESTERN_NORTH (1,162) · test_gold: hand-digitized (328) |
| Model | SegFormer-B3 (MiT-B3 encoder, `smp.Segformer(encoder_name='mit_b3')`, ImageNet pretrained) — 44.6M params (44.1M encoder / 0.5M decoder) |
| Loss | Dice + BCE |
| Metrics | IoU, Precision, Recall, F1 (val-tuned decision threshold) |
| GPU | T4/P100 16GB — batch 8 @ 512px with AMP |
| Output | best checkpoint + ONNX export + training curves + prediction previews |

## Recipe (v2 — the fixes that matter)
1. **Strong geometric augmentation** — Affine (scale 0.75–1.25, shift ±5%, rotate ±45°) + flips + mild color. Run 1 (flips only) let a 32M-param model memorize 3.5K chips by epoch 2.
2. **Differential learning rates** — encoder 3e-5, decoder 3e-4, AdamW wd 1e-2 (AdamW standard; 1e-4 was ~100x too low), cosine schedule.
3. **Early stopping** — patience 7 on val IoU.
4. **Threshold sweep on val** (0.30–0.70) — best threshold applied to test/gold.
5. **Repaired test_gold eval** — gold masks digitize only ~10% of buildings per chip (verified: median gold/overture building-frac ratio = 0.10 across all 328 gold chips; 0/328 ≥ 0.7). Reports recall vs hand labels + IoU/P/R/F1 vs gold ∪ Overture. Naive gold-only IoU printed only as a documented artifact.
6. **AMP (fp16 autocast + GradScaler)** — speed + headroom for B3's ~47M params at batch 8.
7. **ONNX export with `dynamo=False`** — torch 2.10's dynamo exporter needs `onnxscript` (crashed run 1).

**Kaggle setup:** Enable **GPU** and **Internet**. Attach the `ghana-buildings-patches-v2` dataset (from `ghana_buildings_patches_v2.zip`). Cell 2 auto-finds the manifest, so it also works if the old v1 dataset is attached by mistake — but verify the printed chip counts: **train 3,272 / val 424 = v2** (v1 = 3,534/358).

**Why v2 split:** v1's val (AHAFO only) had half the train building density (median frac 0.109 vs 0.211) — a distribution-shifted holdout that made val IoU peak at epoch 1 and decline in BOTH run-1 (U-Net) and run-2 (SegFormer). v2 val = ~10% of 2048px spatial super-blocks per train region, boundary chips dropped → zero train/val pixel overlap, matched densities (train 0.200 / val 0.176 / test 0.183).

**References:** run-1 U-Net val 0.4495 · test 0.4633; run-2 SegFormer val 0.4613 (both = epoch-1 ckpt, then collapse). Anything ≥ 0.55 val, healthy past epoch 5, means the fix worked. If it STILL collapses on the matched-density val → suspect label noise (Overture) next.

In [ ]:
# Cell 1 — Environment
!pip -q install segmentation_models_pytorch albumentations onnx onnxruntime onnxscript 2>/dev/null | tail -1

import os, json, glob, random, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'
print('device:', DEVICE, '| torch:', torch.__version__, '| amp:', AMP_ENABLED)
if DEVICE.type == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 — Locate the attached dataset (robust: recursive search + auto-unzip)
import zipfile

CANDIDATES = glob.glob('/kaggle/input/**/manifest.json', recursive=True)
if not CANDIDATES:
    # maybe Kaggle kept the upload as a zip — extract it
    for zpath in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        print('extracting', zpath, '...')
        with zipfile.ZipFile(zpath) as z:
            z.extractall('/kaggle/working/patches_extracted')
    CANDIDATES = glob.glob('/kaggle/working/**/manifest.json', recursive=True)

if not CANDIDATES:
    print('--- /kaggle/input contents ---')
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files[:10]:
            print(os.path.join(root, f))
    raise AssertionError(
        'manifest.json not found! In the right panel: Add Input -> Your Datasets -> '
        'ghana-buildings-patches, then re-run this cell.')

DATA_ROOT = os.path.dirname(CANDIDATES[0])
print('DATA_ROOT =', DATA_ROOT)
for s in ['train', 'val', 'test', 'test_gold']:
    n = len(glob.glob(os.path.join(DATA_ROOT, s, 'images', '*.jpg')))
    print(f'  {s:9s}: {n} chips')

In [ ]:
# Cell 3 — Dataset & transforms (strong geometric augmentation — the #1 fix)
IMG_SIZE = 512

train_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.75, 1.25), translate_percent=(-0.05, 0.05),
             rotate=(-45, 45), p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

eval_tf = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

class GhanaChips(Dataset):
    def __init__(self, root, split, transform):
        self.img_dir = os.path.join(root, split, 'images')
        self.msk_dir = os.path.join(root, split, 'masks')
        self.transform = transform
        self.chips = sorted(os.path.splitext(os.path.basename(p))[0]
                            for p in glob.glob(os.path.join(self.img_dir, '*.jpg')))

    def __len__(self):
        return len(self.chips)

    def __getitem__(self, i):
        name = self.chips[i]
        img = np.array(Image.open(os.path.join(self.img_dir, name + '.jpg')).convert('RGB'))
        msk = np.array(Image.open(os.path.join(self.msk_dir, name + '.png')).convert('L'))
        msk = (msk > 127).astype(np.float32)
        t = self.transform(image=img, mask=msk)
        return t['image'], t['mask'].unsqueeze(0), name

class GoldChips(Dataset):
    """test_gold chips. Yields (img, gold_mask, overture_mask, name).
    Every gold chip has a same-name chip in the test split (verified: 328/328),
    so the Overture mask for the identical footprint lives in test/masks.
    Needed for the repaired gold eval (gold ∪ Overture union metric)."""
    def __init__(self, root, transform):
        self.img_dir = os.path.join(root, 'test_gold', 'images')
        self.gold_dir = os.path.join(root, 'test_gold', 'masks')
        self.ov_dir = os.path.join(root, 'test', 'masks')
        self.transform = transform
        self.chips = sorted(os.path.splitext(os.path.basename(p))[0]
                            for p in glob.glob(os.path.join(self.img_dir, '*.jpg')))

    def __len__(self):
        return len(self.chips)

    def __getitem__(self, i):
        name = self.chips[i]
        img = np.array(Image.open(os.path.join(self.img_dir, name + '.jpg')).convert('RGB'))
        gold = np.array(Image.open(os.path.join(self.gold_dir, name + '.png')).convert('L'))
        gold = (gold > 127).astype(np.float32)
        ovp = os.path.join(self.ov_dir, name + '.png')
        ov = np.array(Image.open(ovp).convert('L')) if os.path.exists(ovp) else np.zeros_like(gold)
        ov = (ov > 127).astype(np.float32)
        t = self.transform(image=img, mask=gold)
        return t['image'], t['mask'].unsqueeze(0), torch.from_numpy(ov).unsqueeze(0), name

BATCH = 8
train_ds = GhanaChips(DATA_ROOT, 'train', train_tf)
val_ds   = GhanaChips(DATA_ROOT, 'val',   eval_tf)
test_ds  = GhanaChips(DATA_ROOT, 'test',  eval_tf)
gold_ds  = GoldChips(DATA_ROOT, eval_tf)

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                      num_workers=2, pin_memory=True, drop_last=True)
val_dl   = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
gold_dl  = DataLoader(gold_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f'train={len(train_ds)} val={len(val_ds)} test={len(test_ds)} gold={len(gold_ds)}')

In [ ]:
# Cell 4 — Model (SegFormer-B3), loss, differential-LR optimizer, scheduler, AMP
MODEL_NAME = 'segformer_b3'

model = smp.Segformer(
    encoder_name='mit_b3',   # NOT 'mix_transformer_b3' — that's the HF transformers name
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
).to(DEVICE)

dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=True)
bce_loss = nn.BCEWithLogitsLoss()

def criterion(logits, target):
    return 0.5 * dice_loss(logits, target) + 0.5 * bce_loss(logits, target)

enc_params = [p for n, p in model.named_parameters() if n.startswith('encoder.')]
dec_params = [p for n, p in model.named_parameters() if not n.startswith('encoder.')]
optimizer = torch.optim.AdamW([
    {'params': enc_params, 'lr': 3e-5},
    {'params': dec_params, 'lr': 3e-4},
], weight_decay=1e-2)

EPOCHS = 25
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)

def iou_score(logits, target, thresh=0.5):
    preds = (torch.sigmoid(logits) > thresh).float()
    inter = (preds * target).sum(dim=(1, 2, 3))
    union = ((preds + target) >= 1).float().sum(dim=(1, 2, 3))
    return (inter / (union + 1e-7)).mean().item()

print('params:', sum(p.numel() for p in model.parameters()) / 1e6, 'M')
print('encoder lr 3e-5 | decoder lr 3e-4 | wd 1e-2 | amp:', AMP_ENABLED)

In [ ]:
# Cell 5 — Train (AMP + early stopping, patience 7)
best_val_iou, history = -1.0, []
CKPT = f'{MODEL_NAME}_best.pth'
PATIENCE = 7
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    tr_loss, tr_iou = 0.0, 0.0
    for img, msk, _ in train_dl:
        img, msk = img.to(DEVICE, non_blocking=True), msk.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', enabled=AMP_ENABLED):
            logits = model(img)
            loss = criterion(logits, msk)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        tr_loss += loss.item() * img.size(0)
        with torch.no_grad():
            tr_iou += iou_score(logits.float(), msk) * img.size(0)
    tr_loss /= len(train_ds); tr_iou /= len(train_ds)

    model.eval()
    va_loss, va_iou = 0.0, 0.0
    with torch.no_grad():
        for img, msk, _ in val_dl:
            img, msk = img.to(DEVICE, non_blocking=True), msk.to(DEVICE, non_blocking=True)
            with torch.autocast('cuda', enabled=AMP_ENABLED):
                logits = model(img)
                loss = criterion(logits, msk)
            va_loss += loss.item() * img.size(0)
            va_iou += iou_score(logits.float(), msk) * img.size(0)
    va_loss /= len(val_ds); va_iou /= len(val_ds)
    scheduler.step()

    if va_iou > best_val_iou:
        best_val_iou = va_iou
        torch.save(model.state_dict(), CKPT)
        star = ' *saved*'
        bad_epochs = 0
    else:
        star = ''
        bad_epochs += 1
    history.append((tr_loss, tr_iou, va_loss, va_iou))
    print(f'epoch {epoch:02d}/{EPOCHS} [{time.time()-t0:5.0f}s] '
          f'train loss {tr_loss:.4f} IoU {tr_iou:.4f} | '
          f'val loss {va_loss:.4f} IoU {va_iou:.4f}{star}')
    if bad_epochs >= PATIENCE:
        print(f'\nearly stopping at epoch {epoch} (no val improvement for {PATIENCE} epochs)')
        break

print(f'\nbest val IoU: {best_val_iou:.4f} -> {CKPT}')

In [ ]:
# Cell 6 — Threshold sweep on val, then evaluate test + repaired gold eval
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()

THRESHOLDS = [0.30, 0.40, 0.50, 0.60, 0.70]

@torch.no_grad()
def sweep(loader):
    stats = {t: [0.0, 0.0] for t in THRESHOLDS}  # inter, union
    for img, msk, _ in loader:
        img, msk = img.to(DEVICE), msk.to(DEVICE)
        with torch.autocast('cuda', enabled=AMP_ENABLED):
            probs = torch.sigmoid(model(img)).float()
        for t in THRESHOLDS:
            preds = (probs > t).float()
            stats[t][0] += (preds * msk).sum().item()
            stats[t][1] += ((preds + msk) >= 1).float().sum().item()
    return {t: i / (u + 1e-7) for t, (i, u) in stats.items()}

print('--- val threshold sweep (IoU) ---')
val_by_t = sweep(val_dl)
for t, v in val_by_t.items():
    print(f'  thr {t:.2f}: {v:.4f}')
BEST_T = max(val_by_t, key=val_by_t.get)
print(f'best threshold on val: {BEST_T:.2f}')

@torch.no_grad()
def evaluate(loader, name, thresh):
    inter_sum, union_sum, pred_sum, tgt_sum = 0.0, 0.0, 0.0, 0.0
    per_region = {}
    for img, msk, names in loader:
        img, msk = img.to(DEVICE), msk.to(DEVICE)
        with torch.autocast('cuda', enabled=AMP_ENABLED):
            probs = torch.sigmoid(model(img)).float()
        preds = (probs > thresh).float()
        for b in range(img.size(0)):
            p, g = preds[b], msk[b]
            inter = (p * g).sum().item()
            union = ((p + g) >= 1).float().sum().item()
            inter_sum += inter; union_sum += union
            pred_sum += p.sum().item(); tgt_sum += g.sum().item()
            region = names[b].rsplit('_', 2)[0]
            d = per_region.setdefault(region, [0.0, 0.0])
            d[0] += inter; d[1] += union
    iou = inter_sum / (union_sum + 1e-7)
    prec = inter_sum / (pred_sum + 1e-7)
    rec = inter_sum / (tgt_sum + 1e-7)
    f1 = 2 * prec * rec / (prec + rec + 1e-7)
    print(f'[{name}] IoU {iou:.4f} | Precision {prec:.4f} | Recall {rec:.4f} | F1 {f1:.4f} | thr {thresh:.2f}')
    for r, (i, u) in sorted(per_region.items()):
        print(f'    {r:15s} IoU {i/(u+1e-7):.4f}')
    return {'iou': iou, 'precision': prec, 'recall': rec, 'f1': f1, 'threshold': thresh}

@torch.no_grad()
def evaluate_gold(loader, thresh):
    # Gold masks are SPARSE: hand shapefiles digitize only ~10% of buildings per
    # chip (median gold/overture building-frac ratio = 0.10; 0/328 chips >= 0.7).
    # Fair metrics: recall vs gold, and IoU/P/R/F1 vs gold UNION overture.
    inter_g, union_g, pred_sum, gold_sum = 0.0, 0.0, 0.0, 0.0
    inter_u, union_u, tgt_u = 0.0, 0.0, 0.0
    per_region = {}
    for img, gold, ov, names in loader:
        with torch.autocast('cuda', enabled=AMP_ENABLED):
            probs = torch.sigmoid(model(img.to(DEVICE))).float()
        preds = (probs > thresh).float().cpu()
        for b in range(img.size(0)):
            p = preds[b, 0]
            g = gold[b, 0]
            o = (ov[b, 0] > 0.5).float()
            u = ((g + o) >= 1).float()
            inter_g += (p * g).sum().item()
            union_g += ((p + g) >= 1).float().sum().item()
            pred_sum += p.sum().item(); gold_sum += g.sum().item()
            inter_u += (p * u).sum().item()
            union_u += ((p + u) >= 1).float().sum().item()
            tgt_u += u.sum().item()
            region = names[b].rsplit('_', 2)[0]
            d = per_region.setdefault(region, [0.0, 0.0])
            d[0] += (p * u).sum().item(); d[1] += ((p + u) >= 1).float().sum().item()
    rec_gold = inter_g / (gold_sum + 1e-7)
    iou_naive = inter_g / (union_g + 1e-7)
    iou_u = inter_u / (union_u + 1e-7)
    prec_u = inter_u / (pred_sum + 1e-7)
    rec_u = inter_u / (tgt_u + 1e-7)
    f1_u = 2 * prec_u * rec_u / (prec_u + rec_u + 1e-7)
    print('[test_gold — repaired eval, hand-digitized regions]')
    print(f'    recall vs hand labels            {rec_gold:.4f}')
    print(f'    naive gold-only IoU              {iou_naive:.4f}  (label-sparsity artifact — ignore)')
    print(f'    vs gold UNION overture:          IoU {iou_u:.4f} | Precision {prec_u:.4f} | Recall {rec_u:.4f} | F1 {f1_u:.4f}')
    for r, (i, u) in sorted(per_region.items()):
        print(f'        {r:15s} IoU {i/(u+1e-7):.4f}')
    return {'recall_gold': rec_gold, 'iou_gold_naive': iou_naive,
            'iou_union': iou_u, 'precision_union': prec_u,
            'recall_union': rec_u, 'f1_union': f1_u, 'threshold': thresh}

results = {
    'model': MODEL_NAME,
    'best_threshold': BEST_T,
    'val_iou_by_threshold': {f'{t:.2f}': v for t, v in val_by_t.items()},
}
results['test'] = evaluate(test_dl, 'test (Overture labels, unseen regions)', BEST_T)
results['test@0.5'] = evaluate(test_dl, 'test @ 0.50 (run-1 comparability)', 0.5)
results['test_gold'] = evaluate_gold(gold_dl, BEST_T)

with open('segformer_results.json', 'w') as f:
    json.dump(results, f, indent=2)

In [ ]:
# Cell 7 — Visualizations: training curves + prediction vs ground truth grids
model.eval()
os.makedirs('previews', exist_ok=True)
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def show_img(t):
    a = (t * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()
    lo, hi = np.percentile(a, (2, 98))
    return np.clip((a - lo) / (hi - lo + 1e-6), 0, 1)

# 7a — training curves (loss + IoU, train vs val)
if history:
    ep = range(1, len(history) + 1)
    tr_l = [h[0] for h in history]; va_l = [h[2] for h in history]
    tr_i = [h[1] for h in history]; va_i = [h[3] for h in history]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ep, tr_l, label='train'); ax[0].plot(ep, va_l, label='val')
    ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[1].plot(ep, tr_i, label='train'); ax[1].plot(ep, va_i, label='val')
    ax[1].set_title('IoU'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
    fig.tight_layout(); fig.savefig('previews/training_curves.png', dpi=110)
    plt.close(fig)
    print('saved previews/training_curves.png')

# 7b — prediction grids: image | ground truth | prediction (8 samples per split)
N_SHOW = 8
for loader, tag in [(test_dl, 'test'), (gold_dl, 'gold')]:
    batch = []
    names = []
    for b in loader:                            # collect N_SHOW chips from the loader
        batch.append(b)
        names.extend(b[2] if len(b) == 3 else b[3])
        if sum(x[0].size(0) for x in batch) >= N_SHOW:
            break
    img = torch.cat([x[0] for x in batch])[:N_SHOW]
    msk = torch.cat([x[1] for x in batch])[:N_SHOW]
    with torch.no_grad():
        with torch.autocast('cuda', enabled=AMP_ENABLED):
            probs = torch.sigmoid(model(img.to(DEVICE))).float()
        preds = (probs > BEST_T).float().cpu()
    fig, axes = plt.subplots(N_SHOW, 3, figsize=(12, 4 * N_SHOW))
    for b in range(N_SHOW):
        axes[b, 0].imshow(show_img(img[b])); axes[b, 0].set_title(names[b], fontsize=8)
        axes[b, 1].imshow(msk[b, 0], cmap='gray'); axes[b, 1].set_title('ground truth', fontsize=8)
        axes[b, 2].imshow(preds[b, 0], cmap='gray'); axes[b, 2].set_title(f'prediction (thr {BEST_T:.2f})', fontsize=8)
        for ax in axes[b]: ax.set_axis_off()
    fig.suptitle(f'{MODEL_NAME} — {tag}: original vs ground truth vs prediction', y=0.998)
    fig.tight_layout()
    fig.savefig(f'previews/{tag}_predictions.png', dpi=110, bbox_inches='tight')
    plt.close(fig)
    print(f'saved previews/{tag}_predictions.png ({N_SHOW} samples)')

# 7c — inline display
from IPython.display import Image as IPyImage, display
for f in ['previews/training_curves.png', 'previews/test_predictions.png', 'previews/gold_predictions.png']:
    if os.path.exists(f):
        display(IPyImage(f))

In [ ]:
# Cell 7d — POSTPROCESSOR: foggy raw masks -> clean right-angled footprints
# Self-contained (cv2 + shapely). Raw model output overpredicts extent
# (P 0.58 vs R 0.75) -> this cleans it: morphological cleanup + min-area
# filter + watershed instance split + RDP simplify + 90-degree snapping.
# Can be pasted into a LIVE session right after Cell 7 (no retraining).
import cv2
from shapely.geometry import Polygon

GSD = 0.2986  # m/px of the Ghana imagery

def _cleanup_mask(mask, gsd=GSD, min_area_m2=12.0, open_px=3, close_px=3):
    m = (mask > 0).astype(np.uint8) * 255
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_px, open_px)))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_px, close_px)))
    min_px = int(min_area_m2 / (gsd * gsd))
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    out = np.zeros_like(m)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= min_px:
            out[lab == i] = 255
    return out

def _regional_maxima(dist, h):
    marker = (dist.astype(np.float32) - h).clip(min=0)
    rec = marker.copy()
    kernel = np.ones((3, 3), np.uint8)
    for _ in range(500):
        prev = rec
        rec = np.minimum(cv2.dilate(rec, kernel), dist)
        if np.array_equal(rec, prev):
            break
    return rec == cv2.dilate(rec, kernel)

def _split_instances(mask, gsd=GSD, min_dist_m=8.0, min_peak_m=1.5, h_supp_m=2.0, min_area_m2=12.0):
    m = (mask > 0).astype(np.uint8)
    dist = cv2.distanceTransform(m, cv2.DIST_L2, 5).astype(np.float32)
    plateau = _regional_maxima(dist, h_supp_m / gsd) & (dist > min_peak_m / gsd)
    seeds = np.zeros(m.shape, np.uint8); seeds[plateau] = 255
    n, lab = cv2.connectedComponents(seeds, connectivity=8)
    cand = []
    for i in range(1, n):
        ys, xs = np.nonzero(lab == i)
        j = int(np.argmax(dist[ys, xs]))
        cand.append((float(dist[ys[j], xs[j]]), int(ys[j]), int(xs[j])))
    if not cand:
        _, lab = cv2.connectedComponents(m, connectivity=8)
        return lab.astype(np.int32)
    min_dist_px = int(min_dist_m / gsd)
    cand.sort(key=lambda c: -c[0])
    kept = []
    for d, y, x in cand:
        if any(np.hypot(ky - y, kx - x) < min_dist_px for _, ky, kx in kept):
            continue
        kept.append((d, y, x))
    markers = np.zeros(m.shape, np.int32)
    for i, (_, y, x) in enumerate(kept, start=1):
        markers[y, x] = i
    markers = cv2.watershed(cv2.cvtColor(m * 255, cv2.COLOR_GRAY2BGR), markers)
    markers[markers == -1] = 0
    markers[m == 0] = 0
    min_px = int(min_area_m2 / (gsd * gsd))
    out = np.zeros_like(markers)
    for lb in np.unique(markers):
        if lb == 0:
            continue
        comp = markers == lb
        if comp.sum() >= min_px:
            out[comp] = lb
    return out

def _perp_dist(pts, a, b):
    ab = b - a
    n = np.hypot(*ab)
    if n < 1e-12:
        return np.hypot(*(pts - a).T)
    return np.abs(np.cross(ab, pts - a)) / n

def _rdp(points, tol):
    if len(points) < 3:
        return points
    p0, p1 = points[0], points[-1]
    d = _perp_dist(points[1:-1], p0, p1)
    i = int(np.argmax(d)) + 1
    if d[i - 1] > tol:
        left = _rdp(points[:i + 1], tol)
        right = _rdp(points[i:], tol)
        return np.vstack([left[:-1], right])
    return np.vstack([p0, p1])

def _dominant_orientation(poly):
    xy = np.array(poly.exterior.coords)
    dx, dy = np.diff(xy[:, 0]), np.diff(xy[:, 1])
    length = np.hypot(dx, dy)
    keep = length > 0.1
    if not keep.any():
        return 0.0
    ang = (np.degrees(np.arctan2(dy[keep], dx[keep])) % 90.0)
    w = length[keep]
    ca = np.average(np.cos(np.radians(ang)), weights=w)
    sa = np.average(np.sin(np.radians(ang)), weights=w)
    return float(np.degrees(np.arctan2(sa, ca)) % 90.0)

def _orthogonalize(poly, theta=None, snap_tol_deg=15.0):
    if theta is None:
        theta = _dominant_orientation(poly)
    xy = np.array(poly.exterior.coords[:-1], dtype=float)
    n = len(xy)
    if n < 4:
        return poly
    out = [xy[0]]
    for i in range(1, n):
        prev, cur = out[-1], xy[i]
        bearing = np.degrees(np.arctan2(cur[1] - prev[1], cur[0] - prev[0]))
        rel = (bearing - theta) % 90.0
        cand = bearing - rel if rel <= snap_tol_deg else bearing - (rel - 90.0)
        if abs(((cand - bearing + 180) % 360) - 180) <= snap_tol_deg:
            d = float(np.hypot(cur[0] - prev[0], cur[1] - prev[1]))
            rad = np.radians(cand)
            cur = np.array([prev[0] + d * np.cos(rad), prev[1] + d * np.sin(rad)])
        out.append(cur)
    out.append(out[0])
    try:
        p = Polygon(out)
        return p if p.is_valid else poly
    except Exception:
        return poly

def postprocess_mask(mask, gsd=GSD, min_area_m2=12.0):
    m = _cleanup_mask(mask, gsd=gsd, min_area_m2=min_area_m2)
    if m.max() == 0:
        return []
    labels = _split_instances(m, gsd=gsd, min_area_m2=min_area_m2)
    polys = []
    for lb in np.unique(labels):
        if lb == 0:
            continue
        comp = (labels == lb).astype(np.uint8) * 255
        cnts, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in cnts:
            if len(cnt) < 3:
                continue
            pts = cnt.reshape(-1, 2).astype(float) * gsd
            try:
                p = Polygon(pts)
                if not p.is_valid:
                    p = p.buffer(0)
                if p.geom_type == 'MultiPolygon':
                    p = max(p.geoms, key=lambda g: g.area)
                if p.is_empty:
                    continue
            except Exception:
                continue
            tol = 0.02 * p.exterior.length          # RDP simplify
            ring = np.array(p.exterior.coords)[:-1]
            keep = _rdp(ring, tol)
            if len(keep) >= 3:
                keep = np.vstack([keep, keep[0]])
                try:
                    p2 = Polygon(keep)
                    if p2.is_valid:
                        p = p2
                except Exception:
                    pass
            p = _orthogonalize(p)
            if p.is_valid and p.area >= min_area_m2 * 0.5:
                polys.append(p)
    return polys

def polygons_to_mask(polys, shape, gsd=GSD):
    out = np.zeros(shape, np.uint8)
    for p in polys:
        ext = (np.array(p.exterior.coords) / gsd).astype(np.int32)
        cv2.fillPoly(out, [ext], 255)
    return out

# --- run on test + gold samples: IoU before/after + 4-column preview ---
N_SHOW = 8
for loader, tag in [(test_dl, 'test'), (gold_dl, 'gold')]:
    batch = []
    names = []
    for b in loader:
        batch.append(b)
        names.extend(b[2] if len(b) == 3 else b[3])
        if sum(x[0].size(0) for x in batch) >= N_SHOW:
            break
    img = torch.cat([x[0] for x in batch])[:N_SHOW]
    msk = torch.cat([x[1] for x in batch])[:N_SHOW]
    with torch.no_grad():
        with torch.autocast('cuda', enabled=AMP_ENABLED):
            probs = torch.sigmoid(model(img.to(DEVICE))).float()
        raw = (probs > BEST_T).float().cpu().numpy()[:, 0]
    post = np.stack([polygons_to_mask(postprocess_mask(r), r.shape) for r in raw])
    gt = msk.numpy()[:, 0]
    def _iou(a, b):
        inter = np.logical_and(a > 0, b > 0).sum()
        union = np.logical_or(a > 0, b > 0).sum()
        return inter / union if union else 1.0
    iou_raw = np.mean([_iou(r, g) for r, g in zip(raw, gt)])
    iou_post = np.mean([_iou(p, g) for p, g in zip(post, gt)])
    n_bld = [len(postprocess_mask(r)) for r in raw]
    print(f'[{tag}] IoU raw {iou_raw:.4f} -> postprocessed {iou_post:.4f} '
          f'({"+" if iou_post >= iou_raw else ""}{iou_post - iou_raw:.4f}) | '
          f'buildings/chip: {np.mean(n_bld):.1f}')
    fig, axes = plt.subplots(N_SHOW, 4, figsize=(16, 4 * N_SHOW))
    for b in range(N_SHOW):
        axes[b, 0].imshow(show_img(img[b])); axes[b, 0].set_title(names[b], fontsize=8)
        axes[b, 1].imshow(gt[b], cmap='gray'); axes[b, 1].set_title('ground truth', fontsize=8)
        axes[b, 2].imshow(raw[b], cmap='gray'); axes[b, 2].set_title(f'raw prediction (thr {BEST_T:.2f})', fontsize=8)
        axes[b, 3].imshow(post[b], cmap='gray'); axes[b, 3].set_title('postprocessed (clean footprints)', fontsize=8)
        for ax in axes[b]: ax.set_axis_off()
    fig.suptitle(f'{MODEL_NAME} — {tag}: raw vs postprocessed', y=0.998)
    fig.tight_layout()
    fig.savefig(f'previews/{tag}_postprocessed.png', dpi=110, bbox_inches='tight')
    plt.close(fig)
    print(f'saved previews/{tag}_postprocessed.png')

from IPython.display import Image as IPyImage, display
for f in ['previews/test_postprocessed.png', 'previews/gold_postprocessed.png']:
    if os.path.exists(f):
        display(IPyImage(f))

In [ ]:
# Cell 8 — Export ONNX (fAIr deployment artifact)
# dynamo=False: torch 2.10's dynamo exporter needs onnxscript; legacy exporter
# works without it and handles SegFormer fine.
import torch.onnx

model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
ONNX_PATH = f'{MODEL_NAME}_512.onnx'
torch.onnx.export(
    model, dummy, ONNX_PATH,
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
    dynamo=False,
)

import onnxruntime as ort
sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
out = sess.run(None, {'image': dummy.cpu().numpy()})[0]
# sanity: ONNX output vs torch output should match closely
with torch.no_grad():
    ref = model(dummy).cpu().numpy()
diff = np.abs(out - ref).max()
print(f'ONNX export OK: {ONNX_PATH} | output shape {out.shape} | '
      f'max |onnx - torch| {diff:.2e} | size {os.path.getsize(ONNX_PATH)/1e6:.1f} MB')
print('\nArtifacts for Kaggle output:')
for f in [CKPT, ONNX_PATH, 'segformer_results.json'] + glob.glob('previews/*.png'):
    print('  ', f)

## After training

Kaggle keeps everything written to `/kaggle/working` — download:
1. `segformer_b3_best.pth` — checkpoint
2. `segformer_b3_512.onnx` — deployment artifact (goes into the fAIr PR)
3. `segformer_results.json` — metrics incl. best threshold + repaired gold metrics
4. `previews/*.png` — qualitative evidence

**Decision guide:**
- If val IoU ≥ 0.55 and healthy past epoch 5 → recipe validated; this model goes into the fAIr package (step 4 of next steps).
- If val IoU still collapses like run 1 → run the fixed U-Net v2 as a diagnostic control before touching anything else.
- Compare against run-1 U-Net reference: val 0.4495 · test 0.4633 @ thr 0.5.
- `test_gold`: recall vs hand labels = did we find the human-mapped buildings; gold∪overture IoU = fair footprint quality. Ignore the naive gold-only IoU.